# 04 Â· Neural networks


In forward propagation, each dense layer multiplies its inputs by learned weights, adds a bias, and applies an activation. ReLU keeps positive values and maps negative values to zero. He initialization helps keep the scale of the activations reasonable at the start of training. The final sigmoid produces a score between zero and one. Binary crossentropy penalizes scores that assign low probability to the true class. Backpropagation computes gradients of that loss. Adam uses moving averages of gradients and squared gradients to adapt each parameter update.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))

from src.config import TARGET
from src.ingestion import read_data, clean_data, split_data
from src.preprocessing import fit_preprocessor, transform
from src.neural_network import build_mlp, train_networks

splits = split_data(clean_data(read_data()))
train, validation = splits["train"], splits["validation"]
preprocessor = fit_preprocessor(train)
x_train, x_validation = transform(preprocessor, train), transform(preprocessor, validation)
build_mlp(x_train.shape[1]).summary()

Dropout randomly disables some activations during training to help control overfitting. Learning rate controls update size. Wider networks can learn more complex patterns and may overfit. Early stopping restores the epoch with the lowest validation loss after five epochs without improvement. Class weights give the rarer class more influence on the training loss. Compare weighted and unweighted validation ROC-AUC on the baseline, then keep that choice for all six manual configurations. Improvements depend on the data.

In [ ]:
import pandas as pd

# Set quick=True for a shorter classroom run.
models, histories, rows = train_networks(
    x_train, train[TARGET], x_validation, validation[TARGET], quick=False,
)
pd.DataFrame(rows)

Compare learning curves for the two final candidates. The dotted line marks the restored epoch. A .keras file stores the model architecture and weights. A separate joblib file stores the fitted preprocessor. Run train.py to serialize both candidates. Add --export-h5 only if submission needs legacy ANN files.

In [ ]:
from io import BytesIO
from IPython.display import Image, display
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(9, 6))
for (name, history), row in zip(histories.items(), axes):
    for metric, ax in zip(("loss", "accuracy"), row):
        epochs = range(1, len(history[metric]) + 1)
        ax.plot(epochs, history[metric], label="Training")
        ax.plot(epochs, history["val_" + metric], label="Validation")
        ax.axvline(history["best_epoch"], linestyle=":")
        ax.set(title=f"{name} Â· {metric}", xlabel="Epoch")
        ax.legend()
fig.tight_layout()
buffer = BytesIO()
fig.savefig(buffer, format="png")
display(Image(data=buffer.getvalue()))
plt.close(fig)